In [1]:
import os
import requests
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText

model_name = "Qwen/Qwen3-VL-2B-Instruct"

os.makedirs("assets", exist_ok=True)

img1 = "assets/img1.jpg"
img2 = "assets/img2.jpg"
video = "assets/video.mp4"

def download(url, path):
    if not os.path.exists(path):
        r = requests.get(url)
        r.raise_for_status()
        with open(path, "wb") as f:
            f.write(r.content)

download("https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen-VL/assets/demo.jpeg", img1)
download("https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/cats.png", img2)
download("https://www.w3schools.com/html/mov_bbb.mp4", video)

processor = AutoProcessor.from_pretrained(model_name)

model = AutoModelForImageTextToText.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto",
)

def inspect(messages, title):
    print("\n" + "=" * 50)
    print(title)
    print("=" * 50)

    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    print("\nCHAT TEMPLATE:")
    print(text)

    inputs = processor.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)

    print("\nMODEL INPUTS:")
    for k, v in inputs.items():
        if hasattr(v, "shape"):
            print(k, v.shape)
        else:
            print(k, v)

    return inputs


# -------------------------
# Case 1: text only
# -------------------------
inspect(
    [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "Describe what you can do."}
            ],
        }
    ],
    "TEXT ONLY",
)


# -------------------------
# Case 2: two images
# -------------------------
inspect(
    [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": img1},
                {"type": "image", "image": img2},
                {"type": "text", "text": "Compare these two images."},
            ],
        }
    ],
    "TWO IMAGES",
)


# -------------------------
# Case 3: image + video
# -------------------------
inspect(
    [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": img1},
                {"type": "video", "video": video},
                {"type": "text", "text": "Describe the image and video."},
            ],
        }
    ],
    "IMAGE + VIDEO",
)

/home/mv/miniconda3/envs/env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████████████████████████████████████████████████████████████████████████████████████████| 625/625 [00:00<00:00, 733.52it/s]



TEXT ONLY

CHAT TEMPLATE:
<|im_start|>user
Describe what you can do.<|im_end|>
<|im_start|>assistant


MODEL INPUTS:
input_ids torch.Size([1, 14])
attention_mask torch.Size([1, 14])
mm_token_type_ids torch.Size([1, 14])

TWO IMAGES

CHAT TEMPLATE:
<|im_start|>user
<|vision_start|><|image_pad|><|vision_end|><|vision_start|><|image_pad|><|vision_end|>Compare these two images.<|im_end|>
<|im_start|>assistant


MODEL INPUTS:
input_ids torch.Size([1, 3016])
attention_mask torch.Size([1, 3016])
mm_token_type_ids torch.Size([1, 3016])
pixel_values torch.Size([11996, 1536])
image_grid_thw torch.Size([2, 3])

IMAGE + VIDEO

CHAT TEMPLATE:
<|im_start|>user
<|vision_start|><|image_pad|><|vision_end|><|vision_start|><|video_pad|><|vision_end|>Describe the image and video.<|im_end|>
<|im_start|>assistant


MODEL INPUTS:
input_ids torch.Size([1, 3450])
attention_mask torch.Size([1, 3450])
mm_token_type_ids torch.Size([1, 3450])
pixel_values torch.Size([11008, 1536])
image_grid_thw torch.Size([1, 3]

{'input_ids': tensor([[151644,    872,    198,  ..., 151644,  77091,    198]],
       device='cuda:0'), 'attention_mask': tensor([[1, 1, 1,  ..., 1, 1, 1]], device='cuda:0'), 'mm_token_type_ids': tensor([[0, 0, 0,  ..., 0, 0, 0]], device='cuda:0'), 'pixel_values': tensor([[ 0.4196,  0.4196,  0.4275,  ...,  0.5922,  0.5922,  0.5922],
        [ 0.4667,  0.4667,  0.4667,  ...,  0.6235,  0.6235,  0.6235],
        [ 0.4667,  0.4667,  0.4745,  ...,  0.6078,  0.6157,  0.6157],
        ...,
        [-0.1529, -0.1608, -0.1608,  ..., -0.3255, -0.3176, -0.3176],
        [-0.2078, -0.2078, -0.2078,  ..., -0.3333, -0.3412, -0.3490],
        [-0.1765, -0.2000, -0.2235,  ..., -0.4196, -0.4275, -0.4353]],
       device='cuda:0'), 'image_grid_thw': tensor([[  1,  86, 128]], device='cuda:0'), 'pixel_values_videos': tensor([[-0.4745, -0.4431, -0.4353,  ..., -0.4196, -0.4196, -0.4118],
        [-0.2392, -0.2627, -0.3490,  ..., -0.5137, -0.4980, -0.4745],
        [-0.4275, -0.4118, -0.4039,  ..., -0.3647, 

In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen3-0.6B"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto",
)

prompt = "Explain KV cache in simple terms."

messages = [
    {"role": "user", "content": prompt}
]


def inspect_thinking(enable_thinking):
    print("\n" + "=" * 60)
    print("enable_thinking =", enable_thinking)
    print("=" * 60)

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=enable_thinking,
    )

    print("\nCHAT TEMPLATE OUTPUT:")
    print(text)

    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    print("\nMODEL INPUTS:")
    print("input_ids:", inputs["input_ids"].shape)
    print("attention_mask:", inputs["attention_mask"].shape)

    print("\nTOKENS:")
    for i, token_id in enumerate(inputs["input_ids"][0].tolist()):
        token_text = tokenizer.decode([token_id], skip_special_tokens=False)
        print(f"{i:03d} | {token_id:>8} | {repr(token_text)}")


inspect_thinking(enable_thinking=False)
inspect_thinking(enable_thinking=True)

Loading weights: 100%|█████████████████████████████████████████████████████████████████████████████████████████████| 311/311 [00:00<00:00, 2280.39it/s]



enable_thinking = False

CHAT TEMPLATE OUTPUT:
<|im_start|>user
Explain KV cache in simple terms.<|im_end|>
<|im_start|>assistant
<think>

</think>



MODEL INPUTS:
input_ids: torch.Size([1, 20])
attention_mask: torch.Size([1, 20])

TOKENS:
000 |   151644 | '<|im_start|>'
001 |      872 | 'user'
002 |      198 | '\n'
003 |      840 | 'Ex'
004 |    20772 | 'plain'
005 |    84648 | ' KV'
006 |     6500 | ' cache'
007 |      304 | ' in'
008 |     4285 | ' simple'
009 |     3793 | ' terms'
010 |       13 | '.'
011 |   151645 | '<|im_end|>'
012 |      198 | '\n'
013 |   151644 | '<|im_start|>'
014 |    77091 | 'assistant'
015 |      198 | '\n'
016 |   151667 | '<think>'
017 |      271 | '\n\n'
018 |   151668 | '</think>'
019 |      271 | '\n\n'

enable_thinking = True

CHAT TEMPLATE OUTPUT:
<|im_start|>user
Explain KV cache in simple terms.<|im_end|>
<|im_start|>assistant


MODEL INPUTS:
input_ids: torch.Size([1, 16])
attention_mask: torch.Size([1, 16])

TOKENS:
000 |   151644 | '<|im_star